In [ ]:
# 1 Importing Libraries and Data

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')

In [17]:
train = pd.read_csv('DATA\\train.csv')
test = pd.read_csv('DATA\\test.csv')

In [ ]:
# 2 Separating the Sales Price and ID columns

train_id = train['Id']
test_id = test['Id']
y = np.log1p(train['SalePrice'])
train = train.drop(columns=['Id', 'SalePrice'])
test = test.drop(columns=['Id'])

In [ ]:
# 3 Deleting Columns
 

cols_to_drop = ['MiscFeature', 'Fence', 'PoolQC', 'Alley', 'MasVnrType', 'FireplaceQu']
for df in [train, test]:
    df.drop(cols_to_drop, axis=1, inplace=True)

 
# 4 Fill in missing values

obj_col_msn_train = ['BsmtQual', 'BsmtCond', 'BsmtExposure',
                     'BsmtFinType1', 'BsmtFinType2', 'Electrical']

obj_col_msn_test = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd',
                    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                    'BsmtFinType2', 'KitchenQual', 'Functional', 'SaleType']

garage_col = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']

for df, obj_cols in zip([train, test], [obj_col_msn_train, obj_col_msn_test]):
    
    for col in obj_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    if df[num_cols].isnull().any().any():
        imputer = KNNImputer(n_neighbors=5)
        imputed_array = imputer.fit_transform(df[num_cols])
        df[num_cols] = pd.DataFrame(imputed_array, columns=num_cols, index=df.index)

    
    for col in garage_col:
        df[col] = df[col].fillna('NonGarage')

# 5 Ordinal and One-Hot Encoding
 

ordinal_mapping = {
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandContour': ['Low', 'HLS', 'Bnk', 'Lvl'],
    'Utilities': ['NoSeWa', 'AllPub'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'ExterQual': ['Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['Po', 'Fa', 'TA', 'Gd'],
    'BsmtExposure': ['No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['Unf', 'LwQ', 'BLQ', 'Rec', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['Unf', 'BLQ', 'ALQ', 'Rec', 'LwQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'GarageFinish': ['NonGarage', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex', 'NonGarage'],
    'GarageCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex', 'NonGarage'],
    'PavedDrive': ['N', 'P', 'Y']
}

ordinal_cols = list(ordinal_mapping.keys())
ordinal_categories = [ordinal_mapping[col] for col in ordinal_cols]

onehot_cols = [
    'MSZoning', 'Street', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd',
    'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'SaleType', 'SaleCondition'
]

ordinal_enc = OrdinalEncoder(categories=ordinal_categories,
                             handle_unknown='use_encoded_value',
                             unknown_value=-1)

onehot_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)


train[ordinal_cols] = ordinal_enc.fit_transform(train[ordinal_cols])
onehot_encoded_train = onehot_enc.fit_transform(train[onehot_cols])
onehot_feature_names = onehot_enc.get_feature_names_out(onehot_cols)
onehot_df_train = pd.DataFrame(onehot_encoded_train, columns=onehot_feature_names, index=train.index)
train.drop(columns=onehot_cols, inplace=True)
train = pd.concat([train, onehot_df_train], axis=1)


test[ordinal_cols] = ordinal_enc.transform(test[ordinal_cols])
onehot_encoded_test = onehot_enc.transform(test[onehot_cols])
onehot_df_test = pd.DataFrame(onehot_encoded_test, columns=onehot_feature_names, index=test.index)
test.drop(columns=onehot_cols, inplace=True)
test = pd.concat([test, onehot_df_test], axis=1)

# 6 Scaling

scaler = RobustScaler()
train = pd.DataFrame(scaler.fit_transform(train), columns=train.columns, index=train.index)
test = pd.DataFrame(scaler.transform(test), columns=test.columns, index=test.index)


In [ ]:
#  7 Base and Meta model Creation

base_models = [
    ('gb', GradientBoostingRegressor(n_estimators=1000, learning_rate=0.01, max_depth=3, random_state=42)),
    ('xgb', XGBRegressor(n_estimators=1200, learning_rate=0.01, max_depth=4, subsample=0.8, colsample_bytree=0.8, random_state=42)),
    ('lgb', LGBMRegressor(n_estimators=1500, learning_rate=0.01, max_depth=5, num_leaves=32, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)),
    ('cat', CatBoostRegressor(iterations=2000, learning_rate=0.01, depth=5, l2_leaf_reg=3, silent=True, random_state=42)),
    ('ridge', Ridge(alpha=5.0, random_state=42)),
    ('lasso', Lasso(alpha=0.0005, random_state=42)),
    ('elasticnet', ElasticNet(alpha=0.0005, l1_ratio=0.5, random_state=42)),
    ('lr', LinearRegression())
]

meta_model = Ridge(alpha=0.1)
 

#  8 Preparing the dataset, training the base model with k-fold, and predicting with the meta model. 

X_train = train.copy()
y_train = y.copy()
X_test = test.copy()

train_stack = np.zeros((X_train.shape[0], len(base_models)))
test_stack = np.zeros((X_test.shape[0], len(base_models)))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

 
base_results = []

for i, (name, model) in enumerate(base_models):
    print(f"Model: {name}")
    test_pred_fold = np.zeros((X_test.shape[0], kf.get_n_splits()))
    rmses, maes, r2s = [], [], []
    
    for j, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        val_pred = model.predict(X_val)
        train_stack[val_idx, i] = val_pred
        test_pred_fold[:, j] = model.predict(X_test)
        
         
        rmses.append(np.sqrt(mean_squared_error(y_val, val_pred)))
        maes.append(mean_absolute_error(y_val, val_pred))
        r2s.append(r2_score(y_val, val_pred))
    
     
    test_stack[:, i] = test_pred_fold.mean(axis=1)
    
     
    base_results.append({
        'Model': name,
        'RMSE': np.mean(rmses),
        'MAE': np.mean(maes),
        'R2': np.mean(r2s)
    })

meta_model.fit(train_stack, y_train)
final_pred = meta_model.predict(test_stack)

meta_pred_train = meta_model.predict(train_stack)
meta_rmse = np.sqrt(mean_squared_error(y_train, meta_pred_train))
meta_mae = mean_absolute_error(y_train, meta_pred_train)
meta_r2 = r2_score(y_train, meta_pred_train)

#  9 Creating a Results DataFrame

base_df = pd.DataFrame(base_results)

meta_df = pd.DataFrame([{
    'Model': 'meta_model',
    'RMSE': meta_rmse,
    'MAE': meta_mae,
    'R2': meta_r2
}])

results_df = pd.concat([base_df, meta_df], ignore_index=True)
print(results_df)


#  10 Creating a Submission File

final_pred_exp = np.expm1(final_pred)

submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': final_pred_exp
})

submission.to_csv('submission.csv', index=False)
print("END")

Model: gb
Model: xgb
Model: lgb
Model: cat
Model: ridge
Model: lasso
Model: elasticnet
Model: lr
        Model      RMSE       MAE        R2
0          gb  0.131303  0.086225  0.887114
1         xgb  0.127800  0.083510  0.893368
2         lgb  0.129600  0.085152  0.890970
3         cat  0.123687  0.080917  0.900482
4       ridge  0.143798  0.088417  0.854557
5       lasso  0.139408  0.084242  0.860449
6  elasticnet  0.140138  0.084835  0.859695
7          lr  0.146297  0.086579  0.850324
8  meta_model  0.122874  0.079828  0.905312
END
